# ⭐ Star Schema + โหลดเข้า MSSQL

เป้าหมาย: เปลี่ยน `good` เป็น **7 dimensions + 1 fact** แล้ว **โหลดเข้า MSSQL** (schema ของทีม) → จบ ETL!

## เตรียมข้อมูล — ใช้ `loan_etl` ทำ clean/validate/transform ที่เรียนใน 02 ให้เร็ว
(โค้ดชุดเดียวกับที่เขียนเอง แต่ห่อเป็น package ใช้ซ้ำได้)

In [1]:
import sys
sys.path.append("..")   # ให้ import loan_etl จาก repo root ได้

from loan_etl import extract, clean, validate, transform
df = extract.read_csv("../data/loan_sample.csv")
df = clean.clean(df)
good, bad = validate.split_valid(df)
good = transform.derive(good)
print(f"good = {len(good)} | quarantine = {len(bad)}")

good = 5000 | quarantine = 20


## 1️⃣ Dimensions — แต่ละ dim มี **surrogate key** (เลขประจำตัว)

ลองสร้าง `dim_grade` ด้วยมือก่อน เพื่อเข้าใจหลักการ

In [2]:
dim_grade = (good[["grade", "sub_grade"]]
             .drop_duplicates().sort_values("sub_grade").reset_index(drop=True))
dim_grade.insert(0, "grade_key", dim_grade.index + 1)   # 1,2,3,... = surrogate key
dim_grade.head()

,grade_key,grade,sub_grade
0,1,A,A1
1,2,A,A2
2,3,A,A3
3,4,A,A4
4,5,A,A5


In [3]:
# สร้าง dim ทั้ง 7 ตัวด้วย loan_etl (เวอร์ชัน package)
from loan_etl import dimensions, facts
dims = dimensions.build_all(good)
for name, d in dims.items():
    print(f"{name:18s} {len(d):4d} rows")

dim_date             36 rows
dim_grade            35 rows
dim_purpose          11 rows
dim_loan_status       6 rows
dim_geography        16 rows
dim_borrower        180 rows
dim_term              2 rows


## 2️⃣ Fact — `fact_loan` 1 แถว/สินเชื่อ + คีย์ไปหาทุก dim + ตัวเลขวัดผล

In [4]:
fact = facts.build_fact(good, dims)
print("fact_loan:", fact.shape)
fact.head()

fact_loan: (5000, 20)


,loan_id,date_key,grade_key,purpose_key,status_key,geo_key,borrower_key,term_key,loan_amnt,funded_amnt,int_rate,installment,annual_inc,dti,fico_avg,total_pymnt,total_rec_prncp,recoveries,is_default,profit
0,1000000,201609,11,8,4,14,19,2,20000,20000,15.37,479.69,101265.0,5.11,707.0,28345.89,20000.00,0.0,0,8345.89
1,1000001,201703,11,9,4,3,18,1,25000,25000,13.44,847.66,63931.0,16.86,762.0,29185.45,25000.00,0.0,0,4185.45
2,1000002,201601,10,1,2,5,89,2,8000,8000,12.06,178.20,77311.0,16.63,782.0,4587.99,3432.84,0.0,0,-3412.01
3,1000003,201607,3,7,4,11,112,1,5000,5000,5.97,152.04,41439.0,22.84,722.0,5426.27,5000.00,0.0,0,426.27
4,1000004,201801,20,9,5,16,20,1,25000,25000,18.78,913.62,117788.0,27.69,692.0,15039.05,11431.21,0.0,0,-9960.95


## 3️⃣ ทดสอบโหลดลง SQLite ก่อน (รันได้เลย ไม่ต้องต่อ MSSQL)

In [5]:
import sqlalchemy as sa
eng = sa.create_engine("sqlite:///loan_dw_demo.db")
with eng.begin() as conn:
    for name, d in dims.items():
        d.to_sql(name, conn, if_exists="replace", index=False)
    fact.to_sql("fact_loan", conn, if_exists="replace", index=False)
    bad.to_sql("quarantine_loan", conn, if_exists="replace", index=False)
print("โหลดเข้า SQLite สำเร็จ ✅")

โหลดเข้า SQLite สำเร็จ ✅


## 4️⃣ Reconcile — ตรวจว่าข้อมูลไม่หาย/ไม่เพี้ยน

In [6]:
rows_in = len(good) + len(bad)
sum_match = round(good["funded_amnt"].sum(), 2) == round(fact["funded_amnt"].sum(), 2)
print("good + quarantine =", rows_in)
print("fact rows         =", len(fact), "(ควรเท่ากับ good)")
print("ยอด funded ตรงกัน :", sum_match)
print("RECONCILE PASSED  :", (len(fact) == len(good)) and sum_match)

good + quarantine = 5020
fact rows         = 5000 (ควรเท่ากับ good)
ยอด funded ตรงกัน : True
RECONCILE PASSED  : True


## 5️⃣ โหลดเข้า MSSQL จริง — จบ ETL 🎉

> ทุกคนใช้ login เดียวกัน (`deadmin`) ต่อ `de_loan_dw` แต่ตั้ง **`TEAM_ID`** ให้ต่างกัน
> ตารางจะชื่อ `<TEAM_ID>_fact_loan`, `<TEAM_ID>_dim_*` (กันชนกับเพื่อน)
> แค่ตั้ง `LOAD_TO_MSSQL = True` + แก้ `TEAM_ID` → รันได้เลย

In [7]:
LOAD_TO_MSSQL = False     # 👈 เปลี่ยนเป็น True เมื่อพร้อมโหลดเข้า MSSQL
TEAM_ID = "Your_name"        # 👈 ชื่อ/ทีมของคุณ (นำหน้าชื่อตาราง กันชนกัน)

# --- connection กลาง: ทุกคนใช้ตัวเดียวกัน ---
HOST     = "mssql.minddatatech.com"
DATABASE = "de_loan_dw"
USER     = "deadmin"
PASSWORD = "de@admin2026"

if LOAD_TO_MSSQL:
    # URL.create encodes special chars in the password (@ # etc.) automatically
    url = sa.URL.create("mssql+pymssql", username=USER, password=PASSWORD,
                        host=HOST, port=1433, database=DATABASE)
    eng = sa.create_engine(url)
    with eng.begin() as conn:
        for name, d in dims.items():
            d.to_sql(f"{TEAM_ID}_{name}", conn, if_exists="replace", index=False)
        fact.to_sql(f"{TEAM_ID}_fact_loan", conn, if_exists="replace", index=False, chunksize=5000)
        bad.to_sql(f"{TEAM_ID}_quarantine_loan", conn, if_exists="replace", index=False)
    print(f"โหลดเข้า {DATABASE} (ตาราง {TEAM_ID}_*) สำเร็จ ✅ — พร้อมต่อ Power BI")
else:
    print("LOAD_TO_MSSQL = False -> ข้ามขั้น MSSQL (ตอนสอนจริงตั้งเป็น True)")

โหลดเข้า de_loan_dw (ตาราง cc_*) สำเร็จ ✅ — พร้อมต่อ Power BI


---
## ✅ จบกระบวนการ ETL!
ข้อมูลอยู่ใน **star schema** บน MSSQL (`teamXX.fact_loan`, `teamXX.dim_*`) พร้อมให้ **Power BI** ดึงไปทำ dashboard ตอนบ่าย

> 💡 ทั้ง pipeline (clean→validate→transform→dims→fact→load→reconcile) ทำได้ในบรรทัดเดียวด้วย package:

In [74]:
from loan_etl import Settings, run_pipeline
s = Settings()
s.source_csv = "../data/loan_sample.csv"
s.sqlite_path = "loan_dw_demo.db"
report = run_pipeline(s)     # default target = sqlite
print("reconcile_passed:", report["reconcile_passed"], "| fact rows:", report["rows_fact"])

reconcile_passed: True | fact rows: 5000
